# *Import Libraries and Load Dataset*

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv


In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv(
    '/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv',
    encoding='latin1'
)

conn = sqlite3.connect('superstore.db')

df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

9994

# Step 1: Create Tables
**Customers Table**

In [3]:
conn.execute("DROP TABLE IF EXISTS customers")

query = """
CREATE TABLE customers AS
SELECT DISTINCT
    [Customer ID],
    [Customer Name],
    Segment
FROM superstore_raw;
"""

conn.execute(query)
conn.commit()

**Orders Table**

In [4]:
conn.execute("DROP TABLE IF EXISTS orders")

query = """
CREATE TABLE orders AS
SELECT DISTINCT
    [Order ID],
    [Order Date],
    [Ship Date],
    [Ship Mode],
    [Customer ID],
    Sales,
    Quantity,
    Discount,
    Profit
FROM superstore_raw;
"""

conn.execute(query)
conn.commit()

**Products Table**

In [5]:
conn.execute("DROP TABLE IF EXISTS products")

query = """
CREATE TABLE products AS
SELECT DISTINCT
    [Product ID],
    [Product Name],
    Category,
    [Sub-Category]
FROM superstore_raw;
"""

conn.execute(query)
conn.commit()

**Verify Tables**

In [6]:
pd.read_sql("SELECT * FROM customers LIMIT 5", conn)

,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer


In [7]:
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.9400,3,0.00,219.5820
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,14.6200,2,0.00,6.8714
3,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,957.5775,5,0.45,-383.0310
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,22.3680,2,0.20,2.5164


In [8]:
pd.read_sql("SELECT * FROM products LIMIT 5", conn)

,Product ID,Product Name,Category,Sub-Category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


# Step 2: Required Queries
**Orders with Sales Greater than Average Sales (Subquery)**

In [6]:
query = """
SELECT *
FROM superstore_raw
WHERE Sales >
(
    SELECT AVG(Sales)
    FROM superstore_raw
);
"""

pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
3,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152
4,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2355,9974,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,Anne Pryor,Home Office,United States,Los Angeles,...,90032,West,TEC-PH-10004080,Technology,Phones,Avaya 5410 Digital phone,271.9600,5,0.20,27.1960
2356,9977,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,Anne Pryor,Home Office,United States,Los Angeles,...,90032,West,TEC-PH-10002496,Technology,Phones,Cisco SPA301,249.5840,2,0.20,31.1980
2357,9980,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,Anne Pryor,Home Office,United States,Los Angeles,...,90032,West,OFF-BI-10002026,Office Supplies,Binders,Ibico Recycled Linen-Style Covers,437.4720,14,0.20,153.1152
2358,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932


**Highest Sales Order for Each Customer (Subquery)**

In [7]:
query = """
SELECT *
FROM superstore_raw s1
WHERE Sales =
(
    SELECT MAX(Sales)
    FROM superstore_raw s2
    WHERE s1.[Customer ID] = s2.[Customer ID]
);
"""

pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
1,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
2,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092
3,25,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,Emily Burns,Consumer,United States,Orem,...,84057,West,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1044.6300,3,0.00,240.2649
4,28,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,19140,East,FUR-BO-10004834,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,7,0.50,-1665.0522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
790,9926,CA-2015-159534,3/20/2015,3/23/2015,First Class,DH-13075,Dave Hallsten,Corporate,United States,New York City,...,10035,East,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,1087.9360,8,0.20,353.5792
791,9930,CA-2016-129630,9/4/2016,9/4/2016,Same Day,IM-15055,Ionia McGrath,Consumer,United States,San Francisco,...,94122,West,TEC-CO-10003763,Technology,Copiers,Canon PC1060 Personal Laser Copier,2799.9600,5,0.20,944.9865
792,9949,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,...,46203,Central,OFF-AP-10002945,Office Supplies,Appliances,Honeywell Enviracaire Portable HEPA Air Cleane...,2405.2000,8,0.00,793.7160
793,9969,CA-2017-153871,12/11/2017,12/17/2017,Standard Class,RB-19435,Richard Bierner,Consumer,United States,Plainfield,...,7060,East,OFF-BI-10004600,Office Supplies,Binders,Ibico Ibimaster 300 Manual Binding System,735.9800,2,0.00,331.1910


**Calculate Total Sales for Each Customer (CTE)**

In [8]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales DESC;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
788,Roy Skaria,22.328
789,Mitch Gastineau,16.739
790,Carl Jackson,16.520
791,Lela Donovan,5.304


**Customers Whose Total Sales Are Above Average (CTE + Subquery)**

In [9]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT *
FROM customer_sales

WHERE Total_Sales >
(
    SELECT AVG(Total_Sales)
    FROM customer_sales
)

ORDER BY Total_Sales DESC;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
289,Julie Kriz,2932.484
290,Shaun Weien,2921.544
291,Maris LaWare,2921.500
292,Rob Dowd,2912.894


**Rank All Customers Based on Total Sales (Window Function)**

In [10]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT
    [Customer Name],
    Total_Sales,

    RANK() OVER
    (
        ORDER BY Total_Sales DESC
    ) AS Customer_Rank

FROM customer_sales;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales,Customer_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


**Assign Row Numbers to Each Order Within a Customer (PARTITION BY)**

In [11]:
query = """
SELECT
    [Customer Name],
    [Order ID],
    Sales,

    ROW_NUMBER() OVER
    (
        PARTITION BY [Customer Name]
        ORDER BY Sales DESC
    ) AS Order_Row_Number

FROM superstore_raw;
"""

pd.read_sql(query, conn)

,Customer Name,Order ID,Sales,Order_Row_Number
0,Aaron Bergman,CA-2016-140935,341.960,1
1,Aaron Bergman,CA-2014-156587,242.940,2
2,Aaron Bergman,CA-2016-140935,221.980,3
3,Aaron Bergman,CA-2014-156587,48.712,4
4,Aaron Bergman,CA-2014-156587,17.940,5
...,...,...,...,...
9989,Zuschuss Donatelli,CA-2017-141481,61.440,5
9990,Zuschuss Donatelli,CA-2014-143336,22.720,6
9991,Zuschuss Donatelli,US-2016-147991,16.720,7
9992,Zuschuss Donatelli,CA-2016-152471,15.984,8


**Display Top 3 Customers Based on Total Sales**

In [12]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
),

ranked_customers AS
(
    SELECT
        [Customer Name],
        Total_Sales,

        RANK() OVER
        (
            ORDER BY Total_Sales DESC
        ) AS Customer_Rank

    FROM customer_sales
)

SELECT *
FROM ranked_customers
WHERE Customer_Rank <= 3;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales,Customer_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3


# Step 3: Final Combined Query
**JOIN + CTE + Window Function**

In [13]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer ID],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer ID]
)

SELECT
    c.[Customer Name],
    cs.Total_Sales,

    RANK() OVER
    (
        ORDER BY cs.Total_Sales DESC
    ) AS Customer_Rank

FROM customer_sales cs

JOIN customers c
ON cs.[Customer ID] = c.[Customer ID]

ORDER BY Customer_Rank;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales,Customer_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


In [17]:
query = """
SELECT
    [Customer Name],
    COUNT(DISTINCT [Order ID]) AS Orders_Count

FROM superstore_raw

GROUP BY [Customer Name]

HAVING COUNT(DISTINCT [Order ID]) = 1;
"""

pd.read_sql(query, conn)

,Customer Name,Orders_Count
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


In [18]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT *
FROM customer_sales

WHERE Total_Sales >
(
    SELECT AVG(Total_Sales)
    FROM customer_sales
)

ORDER BY Total_Sales DESC;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
289,Julie Kriz,2932.484
290,Shaun Weien,2921.544
291,Maris LaWare,2921.500
292,Rob Dowd,2912.894


# Mini Project
**Top 5 Customers**

In [14]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571


**Bottom 5 Customers**

In [15]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales ASC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Thais Sissman,4.833
1,Lela Donovan,5.304
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,22.328


**Customers Who Made Only One Order**

In [16]:
query = """
SELECT
    [Customer Name],
    COUNT(DISTINCT [Order ID]) AS Orders_Count

FROM superstore_raw

GROUP BY [Customer Name]

HAVING COUNT(DISTINCT [Order ID]) = 1;
"""

pd.read_sql(query, conn)

,Customer Name,Orders_Count
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


**Customers with Above-Average Sales**

In [17]:
query = """
WITH customer_sales AS
(
    SELECT
        [Customer Name],
        SUM(Sales) AS Total_Sales
    FROM superstore_raw
    GROUP BY [Customer Name]
)

SELECT *
FROM customer_sales

WHERE Total_Sales >
(
    SELECT AVG(Total_Sales)
    FROM customer_sales
)

ORDER BY Total_Sales DESC;
"""

pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
...,...,...
289,Julie Kriz,2932.484
290,Shaun Weien,2921.544
291,Maris LaWare,2921.500
292,Rob Dowd,2912.894


**Highest Order Value Per Customer**

In [18]:
query = """
SELECT
    [Customer Name],
    MAX(Sales) AS Highest_Order_Value

FROM superstore_raw

GROUP BY [Customer Name]

ORDER BY Highest_Order_Value DESC;
"""

pd.read_sql(query, conn)

,Customer Name,Highest_Order_Value
0,Sean Miller,22638.480
1,Tamara Chand,17499.950
2,Raymond Buch,13999.960
3,Tom Ashbrook,11199.968
4,Hunter Lopez,10499.970
...,...,...
788,Carl Jackson,16.520
789,Mitch Gastineau,12.320
790,Roy Skaria,9.648
791,Lela Donovan,5.304


# Final Insights

**Customer Sales Insights**

1. A small group of customers contributes a significant portion of total sales.
2. Several customers have only one order and may need engagement strategies.
3. Customers with above-average sales represent high-value business opportunities.
4. Window functions help rank and analyze customer performance efficiently.
5. CTEs simplify complex aggregation logic and improve query readability.
6. Splitting data into customers, orders, and products tables improves database organization and analysis.